In [ ]:
import kagglehub
import pandas as pd
import torch
import torch.nn as nn
from torch.optim import AdamW
import matplotlib.pyplot as plt
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.preprocessing import OneHotEncoder
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import kagglehub
import os
from tqdm import tqdm

In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q1-ka-ai-2026")

print("Path to dataset files:", path)

In [ ]:
# Task 1: Write your code here:
df = pd.read_csv(f"{path}/Q1_data.csv")

In [ ]:
# Task 2: Write your code here: Inspect the first few rows using head()
df.head()


In [ ]:
# Task 3: Write your code here: Display dataset information using info()
df.info()


In [ ]:
# Task 4: Write your code here: Show statistical description using describe()
df.describe()


In [ ]:
# Task 5: Write your code here: Plot the target distribution (delivery_time)
plt.figure(figsize=(10, 5))
plt.hist(df['Delivery_Time'].dropna(), bins=50, edgecolor='black')
plt.title('delivery time Distribution')
plt.xlabel('delivery time')
plt.ylabel('Frequency')
plt.show()

In [ ]:
# Task 1: Write your code here: Drop the 'Order_ID' column from the data
df_clean = df.drop(columns=['Order_ID'])

df_clean.head()

In [ ]:
# Task 2: Write your code here: Handle missing values appropriately (Hint: I guess you want to have a closer look at the columns with missing values :) )

for col in ["Weather", "Traffic_Level", "Time_of_Day", "Vehicle_Type"]:
    df_clean[col] = df_clean[col].fillna(df[col].mode()[0])

for col in ["Courier_Experience_yrs" , "Delivery_Time"]:
    df_clean[col] = df_clean[col].fillna(df[col].mean())

df_clean.info()

In [ ]:
# Task 3: Write your code here: Check and remove duplicates if any exist
def check_duplicates(df):
  duplicates = df.duplicated().sum()
  print(f"Number of Duplicate Samples: {duplicates}")
  if duplicates > 0:
    print("Dropping Duplicates...")
    df.drop_duplicates(inplace=True)
    print("Duplicates Dropped.")
  else:
    print("No Duplicate Samples Found.")

check_duplicates(df_clean)


In [ ]:
df_clean.info()

In [ ]:
# Task 4: Write your code here: Encode categorical variables if needed (Bonus if used One Hot Encoding)
categories = ["Weather", "Traffic_Level", "Time_of_Day", "Vehicle_Type"]

print('data before encoding:\n', categories) #show before encoding


for col in categories:
    le = LabelEncoder()
    df_clean[col] = le.fit_transform(df_clean[col].astype(str))

df_clean.head()


In [ ]:
# Task 5: Write your code here: Apply feature scaling for all features (Use StandardScaler)
scaler = StandardScaler()
standard_scaler = StandardScaler() # Instantiate StandardScaler
df_cleans = standard_scaler.fit_transform(df_clean) # Apply fit_transform


df1= pd.DataFrame(df_cleans, columns=df_clean.columns)
df1.head()

In [ ]:
# Task 6: Write your code here: Check for target imbalance and state if it is imbalanced or not (keep this cell empty if not needed)


In [ ]:
# Task 1: Write your code here:
X = df_clean.drop("Delivery_Time", axis=1).astype(float)
y = df_clean['Delivery_Time'].astype(float)



In [ ]:
# Task 2,3,4,5: Write your code here:
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error as sklearn_mse, mean_absolute_error, r2_score
from sklearn.ensemble import RandomForestRegressor

skf = KFold(n_splits=5, shuffle=True, random_state=42)
skf.split(X, y)

sklearn_models = {
  "Random Forest": RandomForestRegressor(
      n_estimators=320,  # Number of trees
      max_depth=4
  )
}


lr_mse = []
lr_rmse = []
lr_r2 = []
mae = []
model1 = 0
for name in sklearn_models:
  for fold_idx, (train_index, test_index) in enumerate(skf.split(X, y)):
    print(f"\nFold {fold_idx + 1}/{5}")

    # 1. Split data
    X_train, X_test = X.iloc[train_index], X.iloc[test_index]
    y_train, y_test = y.iloc[train_index], y.iloc[test_index]

    # 2. Train & Validate sklearn models
    for model_name, model in sklearn_models.items():
      print(f"Training {model_name}...")
      model1= model.fit(X_train, y_train) # train
      y_pred = model.predict(X_test) # validate

      # 3. Save metrics for that model in this fold
      mse = sklearn_mse(y_test, y_pred)
      rmse = np.sqrt(mse)
      r2 = r2_score(y_test, y_pred)
      mae1 = mean_absolute_error(y_test, y_pred)

      lr_mse.append(mse)
      lr_rmse.append(rmse)
      lr_r2.append(r2)
      mae.append(mae1)



In [ ]:
np.mean(mae)

In [ ]:
# Task 1: Write your code here:
model1.feature_importances_

importances = {}

importances['Random Forest'] = model1.feature_importances_

# Create a 1x3 plot
fig, axes = plt.subplots(1, 3, figsize=(20, 6))
axes = axes.flatten()
features = X.columns

for i, (model_name, imp) in enumerate(importances.items()):
  # Sort features by importance for a cleaner plot
  if i==0:
    sorted_idx = np.argsort(imp)
    ax = axes[i]
    ax.barh(features[sorted_idx], imp[sorted_idx])
    ax.set_title(f"{model_name} Feature Importance")
    ax.set_xlabel("Importance Score")

plt.tight_layout()
plt.show()

In [ ]:
# Task 2: Write your code here:


In [ ]:
# Task Bonus: Write your code here: